# Hybrid Matching Engine — Model Training Notebook

This notebook is designed to run in **Google Colab** (GPU optional, CPU works fine
for the model sizes used here). Copy/upload it into the Drive folder for this
project and run top-to-bottom.

## What this notebook produces

A fitted, exportable set of weights that combine three sub-scores into one
final match probability, replacing the hardcoded weights
`w1=0.45 (BERT similarity), w2=0.40 (graph skill overlap), w3=0.15 (growth potential)`
from the technical design with weights *learned* from labelled data.

## Data sources used

1. **Primary — synthetic Kenyan candidates & jobs** (`data_pipeline/generate_synthetic_data.py`
   in the repo). Roles covered: Data Analyst, DevOps Engineer, Software Engineer,
   Accountant, Business Analyst. This is the actual domain data for the project.
2. **Auxiliary — Kaggle "Profile Matching and Recommendation Dataset"**
   (`users.csv` + `feedback.csv`, MIT licensed, synthetic, generic profile data —
   *not* job/skill specific). It has real accept/reject labels
   (`feedback.action`), which the synthetic job/candidate data does not have on
   its own. It is used **only** to sanity-check that the score-combination
   classifier training methodology works on a labelled dataset with the same
   shape (numeric features → binary accept/reject), before trusting it on the
   rule-based weak labels generated from the synthetic project data. It is not
   presented as job-matching ground truth in the report.
3. Skill weights mirror the Neo4j ontology seed already in
   `backend/app/db/neo4j_db.py` (`RELATED_TO` / `SKILL_REQUIRED` edges), so the
   graph-overlap feature computed here is consistent with what `graph_service.py`
   will compute at request time in Week 6.

## Note on internet/API access

Kaggle downloads require **your own** `kaggle.json` API token (Kaggle account →
Settings → Create New Token). Paste it when prompted in the Data Access section
below — it is never committed to the repo. If you skip that step, the notebook
still runs end-to-end using only the synthetic project data.


## 1. Environment Setup and Reproducibility

Detects whether we are on Colab, installs pinned dependencies only if needed,
fixes all random seeds, and defines the run's constants and output folders.


In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Pinned versions so the notebook is reproducible on a fresh Colab runtime.
    %pip install -q sentence-transformers==3.2.1 scikit-learn==1.5.2 pandas==2.2.3 \
        numpy==1.26.4 matplotlib==3.9.2 kaggle==1.6.17

import json
import random
import re
import warnings
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning)

RUN_SEED = 42
random.seed(RUN_SEED)
np.random.seed(RUN_SEED)

MODEL_VERSION = "hybrid-match-v1"
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    # Edit this to match where you place the project folder inside your Drive.
    PROJECT_ROOT = Path("/content/drive/MyDrive/Hybrid-AI-Workforce-Placement")
else:
    # Running locally (e.g. in VS Code) against the repo's own dataset folder.
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = PROJECT_ROOT / "data_pipeline" / "dataset"
KAGGLE_DIR = PROJECT_ROOT / "data_pipeline" / "dataset" / "kaggle_profile_matching"
ARTIFACTS_DIR = PROJECT_ROOT / "data_pipeline" / "artifacts"
REPORTS_DIR = PROJECT_ROOT / "data_pipeline" / "reports"

for directory in (DATA_DIR, KAGGLE_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print(f"IN_COLAB={IN_COLAB}")
print(f"PROJECT_ROOT={PROJECT_ROOT}")
print(f"DATA_DIR={DATA_DIR}")


## 2. Data Access: Kaggle API Download + Google Drive Paths

Downloads the auxiliary Kaggle dataset if a `kaggle.json` token is available.
This step is **optional** — everything downstream falls back gracefully to the
synthetic project data alone if it's skipped or fails.

To enable it: Kaggle account → *Settings* → *API* → *Create New Token*, which
downloads a `kaggle.json` file. Upload it when prompted below (Colab) or place
it at `~/.kaggle/kaggle.json` (local).


In [ ]:
import os
import shutil
import zipfile

KAGGLE_DATASET_SLUG = "debabratakuiry/profile-matching-and-recommendation-dataset"
KAGGLE_ENABLED = False

kaggle_json_path = Path.home() / ".kaggle" / "kaggle.json"

if IN_COLAB and not kaggle_json_path.exists():
    try:
        from google.colab import files

        print("Upload your kaggle.json (or press Cancel to skip Kaggle download).")
        uploaded = files.upload()
        if "kaggle.json" in uploaded:
            kaggle_json_path.parent.mkdir(parents=True, exist_ok=True)
            kaggle_json_path.write_bytes(uploaded["kaggle.json"])
            os.chmod(kaggle_json_path, 0o600)
    except Exception as exc:  # user cancelled the upload widget, etc.
        print(f"Skipping Kaggle token upload: {exc}")

if kaggle_json_path.exists():
    try:
        import kaggle  # imported lazily: needs KAGGLE_CONFIG_DIR/kaggle.json to exist

        os.environ["KAGGLE_CONFIG_DIR"] = str(kaggle_json_path.parent)
        kaggle.api.authenticate()
        kaggle.api.dataset_download_files(
            KAGGLE_DATASET_SLUG, path=str(KAGGLE_DIR), unzip=False, quiet=False
        )
        zip_path = KAGGLE_DIR / "profile-matching-and-recommendation-dataset.zip"
        if zip_path.exists():
            with zipfile.ZipFile(zip_path) as zf:
                zf.extractall(KAGGLE_DIR)
            zip_path.unlink()
        KAGGLE_ENABLED = (KAGGLE_DIR / "users.csv").exists()
    except Exception as exc:
        print(f"Kaggle download unavailable, continuing without it: {exc}")
        KAGGLE_ENABLED = False
else:
    print("No kaggle.json found — continuing with synthetic project data only.")

print(f"KAGGLE_ENABLED={KAGGLE_ENABLED}")


## 3. Load Core Tables

Loads the project's own synthetic `candidates.json` / `jobs.json`
(generated by `data_pipeline/generate_synthetic_data.py`) plus, if available,
the Kaggle `users.csv` / `feedback.csv` auxiliary tables.

If `data_pipeline/dataset/candidates.json` doesn't exist yet, generate it first:

```bash
python data_pipeline/generate_synthetic_data.py --candidates 800 --jobs 100
```


In [ ]:
candidates_path = DATA_DIR / "candidates.json"
jobs_path = DATA_DIR / "jobs.json"

if not candidates_path.exists() or not jobs_path.exists():
    raise FileNotFoundError(
        "Run `python data_pipeline/generate_synthetic_data.py` first, or copy "
        "candidates.json / jobs.json into data_pipeline/dataset/ (or your Drive "
        "mirror of it) before running this notebook."
    )

candidates_df = pd.json_normalize(json.loads(candidates_path.read_text(encoding="utf-8")))
jobs_df = pd.json_normalize(json.loads(jobs_path.read_text(encoding="utf-8")))

print(f"candidates_df: {candidates_df.shape}")
print(f"jobs_df: {jobs_df.shape}")
display(candidates_df.head(3))
display(jobs_df.head(3))

kaggle_users_df = pd.DataFrame()
kaggle_feedback_df = pd.DataFrame()
if KAGGLE_ENABLED:
    kaggle_users_df = pd.read_csv(KAGGLE_DIR / "users.csv")
    kaggle_feedback_df = pd.read_csv(KAGGLE_DIR / "feedback.csv")
    kaggle_users_df.columns = [c.strip().lower() for c in kaggle_users_df.columns]
    kaggle_feedback_df.columns = [c.strip().lower() for c in kaggle_feedback_df.columns]
    print(f"kaggle_users_df: {kaggle_users_df.shape}")
    print(f"kaggle_feedback_df: {kaggle_feedback_df.shape}")
    display(kaggle_feedback_df.head(3))


## 4. Schema Validation and Data Quality Report

Fails fast (raises) on missing required columns; otherwise reports missing
values, duplicate IDs, and out-of-range numeric fields as a small JSON report
saved to `data_pipeline/reports/`.


In [ ]:
REQUIRED_CANDIDATE_COLS = {
    "candidate_id", "target_role", "specialisation", "location", "years_experience",
    "education", "skills", "certifications", "expected_salary_kes", "resume_text",
    "latent_competence", "latent_adaptability",
}
REQUIRED_JOB_COLS = {
    "job_id", "title", "specialisation", "location", "required_experience_years",
    "required_skills", "mandatory_certifications", "salary_range_max", "description",
    "latent_quality_bar",
}

# Latent fields drive the label only; leaking them into the model would recreate
# exactly the circularity this schema is designed to avoid.
FORBIDDEN_FEATURE_COLS = {"latent_competence", "latent_adaptability", "latent_quality_bar"}


def validate_schema(df: pd.DataFrame, required_cols: set[str], name: str) -> None:
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(
            f"{name} is missing required columns: {sorted(missing)}. "
            "Regenerate with data_pipeline/generate_synthetic_data.py."
        )


validate_schema(candidates_df, REQUIRED_CANDIDATE_COLS, "candidates_df")
validate_schema(jobs_df, REQUIRED_JOB_COLS, "jobs_df")


def quality_report(df: pd.DataFrame, id_col: str) -> dict:
    return {
        "row_count": int(len(df)),
        "duplicate_ids": int(df[id_col].duplicated().sum()),
        "missing_values_by_column": df.isna().sum().to_dict(),
    }


report = {
    "candidates": quality_report(candidates_df, "candidate_id"),
    "jobs": quality_report(jobs_df, "job_id"),
}

# Flag (not fail on) implausible values so they can be reviewed before training.
report["candidates"]["negative_experience_rows"] = int((candidates_df["years_experience"] < 0).sum())
report["jobs"]["negative_experience_rows"] = int((jobs_df["required_experience_years"] < 0).sum())

# Text diversity matters: near-identical descriptions leave the embedding model
# nothing to learn, which is what crippled S_bert in the first training run.
report["candidates"]["distinct_resume_texts"] = int(candidates_df["resume_text"].nunique())
report["jobs"]["distinct_descriptions"] = int(jobs_df["description"].nunique())

(REPORTS_DIR / "data_quality_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
print(json.dumps(report, indent=2))


## 5. Text Normalization for Skills, Titles, and Descriptions

Lowercases, strips punctuation, and normalizes whitespace for the free-text
fields (`resume_text`, `description`); normalizes skill/certification strings
to a consistent casing so they match the Neo4j ontology seed exactly.


In [ ]:
_WHITESPACE_RE = re.compile(r"\s+")
_PUNCT_RE = re.compile(r"[^\w\s]")

STOPWORDS = {
    "a", "an", "the", "and", "or", "of", "in", "on", "at", "to", "for", "with",
    "is", "are", "was", "were", "be", "been", "as", "we", "our", "this", "that",
}


def normalize_text(text: str) -> str:
    text = str(text).lower()
    text = _PUNCT_RE.sub(" ", text)
    text = _WHITESPACE_RE.sub(" ", text).strip()
    tokens = [tok for tok in text.split(" ") if tok and tok not in STOPWORDS]
    return " ".join(tokens)


def normalize_skill_list(skills: list[str]) -> list[str]:
    return [s.strip() for s in skills if isinstance(s, str) and s.strip()]


candidates_df["resume_text_clean"] = candidates_df["resume_text"].apply(normalize_text)
jobs_df["description_clean"] = jobs_df["description"].apply(normalize_text)
candidates_df["skills"] = candidates_df["skills"].apply(normalize_skill_list)
candidates_df["certifications"] = candidates_df["certifications"].apply(normalize_skill_list)
jobs_df["required_skills"] = jobs_df["required_skills"].apply(normalize_skill_list)
jobs_df["mandatory_certifications"] = jobs_df["mandatory_certifications"].apply(normalize_skill_list)

candidates_df[["resume_text", "resume_text_clean"]].head(2)


## 6. Semantic Feature Extraction (Sentence-BERT Embeddings)

Embeds every candidate résumé and job description with a small pretrained
Sentence-BERT model (`all-MiniLM-L6-v2` — 384-d, CPU-friendly, ~90MB). This is
the same family of model `bert_embedder.py` will load in Week 6, so weights
learned here transfer directly.


In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(EMBEDDING_MODEL_NAME)

candidate_embeddings = embedder.encode(
    candidates_df["resume_text_clean"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)
job_embeddings = embedder.encode(
    jobs_df["description_clean"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)

np.save(ARTIFACTS_DIR / "candidate_embeddings.npy", candidate_embeddings)
np.save(ARTIFACTS_DIR / "job_embeddings.npy", job_embeddings)

print(f"candidate_embeddings: {candidate_embeddings.shape}")
print(f"job_embeddings: {job_embeddings.shape}")


## 7. Feature Engineering: Candidate (User) Profile Features

Builds the numeric/categorical side of each candidate profile and keeps the
normalized skill set for graph-overlap scoring. Education is ordinal-encoded
because it has a natural order (Diploma < Bachelor's < Master's).


In [ ]:
EDUCATION_RANK = {"Diploma": 0, "Bachelor's": 1, "Master's": 2}

candidates_df["education_rank"] = candidates_df["education"].map(EDUCATION_RANK).fillna(0)
candidates_df["skill_set"] = candidates_df["skills"].apply(set)
candidates_df["certification_set"] = candidates_df["certifications"].apply(set)
candidates_df["embedding_idx"] = np.arange(len(candidates_df))

candidates_df[[
    "candidate_id", "target_role", "years_experience", "education_rank", "expected_salary_kes",
]].describe()


## 8. Feature Engineering: Job Posting Features

Mirrors the candidate-side feature engineering for job postings, plus a
skill-weight lookup that mirrors the Neo4j ontology seed's `RELATED_TO`
weights, so `S_graph` here matches what the live graph queries will compute.


In [ ]:
# Mirrors backend/app/db/neo4j_db.py's RELATED_TO seed weights.
SKILL_RELATION_WEIGHTS: dict[tuple[str, str], float] = {
    ("Python", "Data Analysis"): 0.8,
    ("SQL", "Data Analysis"): 0.7,
    ("Financial Accounting", "Taxation"): 0.9,
}


def graph_overlap_score(candidate_skills: set[str], required_skills: list[str]) -> float:
    """Weighted overlap ratio: direct matches count 1.0, RELATED_TO matches use the edge weight."""
    if not required_skills:
        return 1.0
    total_weight = 0.0
    matched_weight = 0.0
    for req_skill in required_skills:
        total_weight += 1.0
        if req_skill in candidate_skills:
            matched_weight += 1.0
            continue
        best_related = max(
            (
                SKILL_RELATION_WEIGHTS.get((cand_skill, req_skill))
                or SKILL_RELATION_WEIGHTS.get((req_skill, cand_skill))
                or 0.0
                for cand_skill in candidate_skills
            ),
            default=0.0,
        )
        matched_weight += best_related
    return matched_weight / total_weight


jobs_df["embedding_idx"] = np.arange(len(jobs_df))
jobs_df["required_skill_set"] = jobs_df["required_skills"].apply(set)
jobs_df["mandatory_cert_set"] = jobs_df["mandatory_certifications"].apply(set)

jobs_df[["job_id", "title", "required_experience_years", "salary_range_max"]].describe()


## 9. Candidate Pair Generation and Label Construction

Builds `(candidate, job)` pairs restricted to the same role family (mirrors
Tier-1 filtering — cross-role pairs would swamp the set with trivial negatives).

### Why the label changed

The first training run used
`is_match = (S_graph >= 0.6) AND (years_experience >= required_years)`.
That was **circular**: `S_graph` is one of the model's own input features, and
`S_growth` encodes the experience term. Measured on all 15,996 same-role pairs, a
hand-written two-threshold rule over the model's own features reproduced that
label with **97.5% accuracy** — better than the trained model's 92.1%. The model
was rediscovering the labelling rule, so its 0.979 ROC-AUC said nothing about
matching quality.

The replacement label is a **probabilistic outcome**, with every driver centred on
its own mean so the class balance stays near 50/50:

$$\text{logit} = 6.0\,\Delta S_{graph} + 3.8\,\Delta \text{exp\_fit} + 3.6\,\Delta \text{domain\_match} + 1.6\,\Delta \text{hidden\_edge} + 0.4\,\text{adaptability} + \varepsilon$$

$$\text{is\_match} \sim \text{Bernoulli}(\sigma(\text{logit})), \qquad \varepsilon \sim \mathcal{N}(0, 0.35^2)$$

where `hidden_edge = latent_competence − latent_quality_bar`.

Three properties this buys us:

1. **`domain_match`** (same specialisation or not) carries a large weight but is
   only visible to the model *through the résumé and job text*, so `S_bert` finally
   has something real to contribute rather than being decorative.
2. **`latent_competence`, `latent_quality_bar`, `latent_adaptability`** are hidden
   traits the model never receives, so the label cannot be reverse-engineered from
   the features.
3. **Bernoulli sampling plus Gaussian noise** guarantees irreducible error.

### Calibration

These weights were chosen empirically, not guessed. Measured over the 4,000 pairs:

| Quantity | Old label | New label |
| --- | --- | --- |
| Positive rate | 0.644 | ~0.50 |
| Best 2-threshold rule over the model's own features | **0.975** | **~0.69** |
| Bayes accuracy ceiling | ~1.00 | **~0.80** |

So performance should be read against a **~0.80 ceiling** and a **~0.50 floor**,
not against 1.0. The cell below prints all three numbers for the actual run.


In [ ]:
def experience_fit(years_experience: int, required_experience_years: int) -> float:
    return min(years_experience / max(required_experience_years, 1), 1.5) / 1.5


def growth_score(cert_count: int, years_experience: int, required_experience_years: int) -> float:
    fit = experience_fit(years_experience, required_experience_years)
    cert_bonus = min(cert_count / 3, 1.0)
    return float(np.clip(0.7 * fit + 0.3 * cert_bonus, 0.0, 1.0))


# Label-generation weights, calibrated so that the positive rate sits near 50% and the
# Bayes ceiling lands around 0.80 -- leaving a wide, honest band between "always predict
# the majority class" and "perfect knowledge" for the model to be judged in.
B_GRAPH, B_EXPERIENCE, B_DOMAIN, B_HIDDEN, B_ADAPT = 6.0, 3.8, 3.6, 1.6, 0.4
NOISE_SD = 0.35

MAX_JOBS_PER_CANDIDATE = 5  # same-role jobs sampled per candidate; keeps the pair set tractable
jobs_by_role = {role: df for role, df in jobs_df.groupby("title")}

rows = []
for _, cand in candidates_df.iterrows():
    role_jobs = jobs_by_role.get(cand["target_role"])
    if role_jobs is None or role_jobs.empty:
        continue
    sampled_jobs = role_jobs.sample(
        n=min(MAX_JOBS_PER_CANDIDATE, len(role_jobs)), random_state=RUN_SEED
    )
    for _, job in sampled_jobs.iterrows():
        rows.append(
            {
                "candidate_id": cand["candidate_id"],
                "job_id": job["job_id"],
                "S_bert": float(
                    np.dot(
                        candidate_embeddings[cand["embedding_idx"]],
                        job_embeddings[job["embedding_idx"]],
                    )
                ),
                "S_graph": graph_overlap_score(cand["skill_set"], job["required_skills"]),
                "S_growth": growth_score(
                    len(cand["certification_set"]),
                    cand["years_experience"],
                    job["required_experience_years"],
                ),
                # Label-only inputs, dropped before the model ever sees the table.
                "_exp_fit": experience_fit(
                    cand["years_experience"], job["required_experience_years"]
                ),
                "_domain_match": float(cand["specialisation"] == job["specialisation"]),
                "_hidden_edge": cand["latent_competence"] - job["latent_quality_bar"],
                "_adaptability": cand["latent_adaptability"] - 0.5,
            }
        )

pairs_df = pd.DataFrame(rows)

# Centre each driver on its own mean so the intercept does not bias the class balance.
label_rng = np.random.default_rng(RUN_SEED)
logit = (
    B_GRAPH * (pairs_df["S_graph"] - pairs_df["S_graph"].mean())
    + B_EXPERIENCE * (pairs_df["_exp_fit"] - pairs_df["_exp_fit"].mean())
    + B_DOMAIN * (pairs_df["_domain_match"] - pairs_df["_domain_match"].mean())
    + B_HIDDEN * (pairs_df["_hidden_edge"] - pairs_df["_hidden_edge"].mean())
    + B_ADAPT * pairs_df["_adaptability"]
    + label_rng.normal(0.0, NOISE_SD, len(pairs_df))
)
pairs_df["true_probability"] = 1.0 / (1.0 + np.exp(-logit))
pairs_df["is_match"] = (label_rng.random(len(pairs_df)) < pairs_df["true_probability"]).astype(int)

# Drop every label-only column so latents cannot reach the model by accident.
pairs_df = pairs_df.drop(columns=[c for c in pairs_df.columns if c.startswith("_")])
assert not FORBIDDEN_FEATURE_COLS & set(pairs_df.columns), "Latent fields leaked into the pair table"

pairs_df.to_parquet(ARTIFACTS_DIR / "match_pairs.parquet", index=False)

positive_rate = pairs_df["is_match"].mean()
bayes_ceiling = float(
    np.maximum(pairs_df["true_probability"], 1 - pairs_df["true_probability"]).mean()
)
majority_baseline = max(positive_rate, 1 - positive_rate)
print(f"pairs_df: {pairs_df.shape}, positive rate: {positive_rate:.3f}")
print(f"majority-class baseline : {majority_baseline:.3f}")
print(f"Bayes accuracy ceiling  : {bayes_ceiling:.3f}")
print(f"usable band             : {bayes_ceiling - majority_baseline:.3f}")
pairs_df.head()


## 10. Train/Validation/Test Split with Leakage Controls

Two kinds of leakage matter here, and the first run only controlled one of them.

**Candidate leakage** — each candidate contributes several rows, so splitting by
row would put the same person in train and test. Handled by grouping on
`candidate_id`.

**Job leakage** — in the first run all 100 jobs appeared in train *and* test, so
the reported numbers only described *warm-start* performance. But the real
product scenario is a recruiter posting a **brand-new job**. That case was
completely untested.

So we now hold out a slice of jobs entirely before splitting anything else:

| Split | Candidates | Jobs | Answers |
| --- | --- | --- | --- |
| train | seen | warm | — |
| validation | unseen | warm | threshold / hyperparameter choice |
| test (warm) | unseen | warm | "new candidate, known job" |
| test (cold) | unseen | **unseen** | "new job posting" — the product scenario |

A large warm-vs-cold gap would mean the model is memorising specific postings
rather than learning transferable matching.


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

FEATURE_COLS = ["S_bert", "S_graph", "S_growth"]
COLD_JOB_FRACTION = 0.20

# 1) Hold out whole job postings first, so the cold-start test never shares a job
#    with anything the model trained on.
all_job_ids = pairs_df["job_id"].unique()
cold_job_ids = set(
    pd.Series(all_job_ids).sample(frac=COLD_JOB_FRACTION, random_state=RUN_SEED)
)
warm_pairs = pairs_df[~pairs_df["job_id"].isin(cold_job_ids)].reset_index(drop=True)
cold_pairs = pairs_df[pairs_df["job_id"].isin(cold_job_ids)].reset_index(drop=True)

# 2) Split the warm pool by candidate: 60 / 20 / 20.
splitter = GroupShuffleSplit(n_splits=1, test_size=0.4, random_state=RUN_SEED)
train_idx, temp_idx = next(splitter.split(warm_pairs, groups=warm_pairs["candidate_id"]))
temp_df = warm_pairs.iloc[temp_idx]

val_test_splitter = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=RUN_SEED)
val_idx, test_idx = next(val_test_splitter.split(temp_df, groups=temp_df["candidate_id"]))

train_df = warm_pairs.iloc[train_idx].reset_index(drop=True)
val_df = temp_df.iloc[val_idx].reset_index(drop=True)
test_df = temp_df.iloc[test_idx].reset_index(drop=True)

# 3) Cold-start test = unseen jobs AND candidates the model never trained on.
train_candidates = set(train_df["candidate_id"])
cold_test_df = cold_pairs[~cold_pairs["candidate_id"].isin(train_candidates)].reset_index(drop=True)

val_candidates = set(val_df["candidate_id"])
test_candidates = set(test_df["candidate_id"])
assert not (train_candidates & val_candidates), "Leakage: candidate in train and val"
assert not (train_candidates & test_candidates), "Leakage: candidate in train and test"
assert not (val_candidates & test_candidates), "Leakage: candidate in val and test"
assert not (set(train_df["job_id"]) & cold_job_ids), "Leakage: cold job seen in training"
assert not (set(cold_test_df["candidate_id"]) & train_candidates), "Leakage: candidate in train and cold test"

for name, frame in [("train", train_df), ("val", val_df), ("test (warm)", test_df), ("test (cold)", cold_test_df)]:
    print(f"{name:<12} rows={len(frame):<6} candidates={frame['candidate_id'].nunique():<5} "
          f"jobs={frame['job_id'].nunique():<4} positive rate={frame['is_match'].mean():.3f}")


## 11. Baseline Models and Ranking Metrics

Two baselines to beat:

1. **Fixed-weight rule** — the technical design's hardcoded
   `0.45*S_bert + 0.40*S_graph + 0.15*S_growth`.
2. **S_bert-only** cosine similarity.

### Fair-comparison fix

The first run thresholded both baselines at a flat `0.5` while the learned model
got its threshold tuned on validation. That was not a fair fight: the fixed-weight
score averages ~0.72, so a 0.5 cut-off marked almost everything positive
(recall came out at exactly **1.000** with precision **0.629** ≈ the base rate —
i.e. the "baseline" was essentially a constant classifier).

Every model below — baselines included — now gets its threshold tuned on the
**validation split** by the same procedure, so the comparison reflects the
scoring functions rather than an arbitrary cut-off. ROC-AUC is reported too since
it is threshold-independent and therefore immune to this problem.


In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score

THRESHOLD_GRID = np.linspace(0.05, 0.95, 91)


def tune_threshold(y_true, scores) -> float:
    """Pick the threshold maximising F1 on whatever split is passed in (always validation)."""
    return float(THRESHOLD_GRID[int(np.argmax([f1_score(y_true, scores >= t) for t in THRESHOLD_GRID]))])


def ranking_metrics(df: pd.DataFrame, score_col: str, k: int = 3) -> dict[str, float]:
    precisions, recalls, ndcgs = [], [], []
    for _, group in df.groupby("candidate_id"):
        ranked = group.sort_values(score_col, ascending=False)
        top_k = ranked.head(k)
        relevant_total = group["is_match"].sum()
        if relevant_total == 0:
            continue
        hits = top_k["is_match"].sum()
        precisions.append(hits / min(k, len(ranked)))
        recalls.append(hits / relevant_total)
        gains = top_k["is_match"].to_numpy()
        discounts = 1.0 / np.log2(np.arange(2, len(gains) + 2))
        dcg = float(np.sum(gains * discounts))
        ideal_gains = np.sort(group["is_match"].to_numpy())[::-1][:k]
        idcg = float(np.sum(ideal_gains * discounts[: len(ideal_gains)])) or 1.0
        ndcgs.append(dcg / idcg)
    return {
        f"precision@{k}": float(np.mean(precisions)) if precisions else 0.0,
        f"recall@{k}": float(np.mean(recalls)) if recalls else 0.0,
        f"ndcg@{k}": float(np.mean(ndcgs)) if ndcgs else 0.0,
    }


def fixed_weight_score(df: pd.DataFrame) -> np.ndarray:
    return (0.45 * df["S_bert"] + 0.40 * df["S_graph"] + 0.15 * df["S_growth"]).to_numpy()


def bert_only_score(df: pd.DataFrame) -> np.ndarray:
    return df["S_bert"].to_numpy()


def evaluate_scorer(name, scorer, val_frame, test_frame, k=3) -> dict[str, float]:
    threshold = tune_threshold(val_frame["is_match"], scorer(val_frame))
    test_scores = scorer(test_frame)
    preds = test_scores >= threshold
    scored = test_frame.copy()
    scored[f"_{name}_score"] = test_scores
    return {
        "threshold": threshold,
        "precision": precision_score(test_frame["is_match"], preds, zero_division=0),
        "recall": recall_score(test_frame["is_match"], preds, zero_division=0),
        "f1": f1_score(test_frame["is_match"], preds, zero_division=0),
        "roc_auc": roc_auc_score(test_frame["is_match"], test_scores),
        **ranking_metrics(scored, f"_{name}_score", k=k),
    }


baseline_results = {
    "fixed_weight_rule": evaluate_scorer("fixed", fixed_weight_score, val_df, test_df),
    "bert_only": evaluate_scorer("bert", bert_only_score, val_df, test_df),
}

pd.DataFrame(baseline_results).T


## 12. Advanced Model: Learned Score-Combination (Logistic Regression)

Fits an L2-regularized logistic regression on standardized
`[S_bert, S_graph, S_growth]` to replace the hardcoded weights. Logistic
regression is deliberately used instead of a larger model (gradient boosting,
neural net): with only 3 input features and a few thousand rows, a bigger
model would only add variance without adding signal — the classic
overfitting risk for this feature/row ratio.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

X_train, y_train = train_df[FEATURE_COLS].to_numpy(), train_df["is_match"].to_numpy()
X_val, y_val = val_df[FEATURE_COLS].to_numpy(), val_df["is_match"].to_numpy()
X_test, y_test = test_df[FEATURE_COLS].to_numpy(), test_df["is_match"].to_numpy()

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RUN_SEED)),
])

param_grid = {"clf__C": [0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0]}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RUN_SEED)

cv_train_scores, cv_val_scores = [], []
for C in param_grid["clf__C"]:
    pipeline.set_params(clf__C=C)
    fold_train, fold_val = [], []
    for tr_idx, va_idx in cv.split(X_train, y_train):
        pipeline.fit(X_train[tr_idx], y_train[tr_idx])
        fold_train.append(pipeline.score(X_train[tr_idx], y_train[tr_idx]))
        fold_val.append(pipeline.score(X_train[va_idx], y_train[va_idx]))
    cv_train_scores.append(np.mean(fold_train))
    cv_val_scores.append(np.mean(fold_val))

cv_results_df = pd.DataFrame({
    "C": param_grid["clf__C"],
    "train_accuracy": cv_train_scores,
    "cv_val_accuracy": cv_val_scores,
    "train_val_gap": np.array(cv_train_scores) - np.array(cv_val_scores),
})
display(cv_results_df)

best_C = cv_results_df.loc[cv_results_df["cv_val_accuracy"].idxmax(), "C"]
print(f"Selected C={best_C} by cross-validated accuracy on the training split only.")


## 13. Overfitting/Underfitting Diagnostics

Plots the train-vs-validation accuracy curve across the `C` grid searched
above. Reading it:

- **Underfitting** region: both train and validation accuracy are low and
  close together (too much regularization, `C` too small).
- **Good fit**: validation accuracy near its peak, small train/validation gap.
- **Overfitting** region: train accuracy keeps climbing while validation
  accuracy flattens or drops, and the gap widens (`C` too large).

A learning curve (accuracy vs. training-set size, at the selected `C`) is
also plotted to confirm more data would help, if it doesn't already.


In [ ]:
import matplotlib.pyplot as plt
from sklearn.model_selection import learning_curve

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(cv_results_df["C"], cv_results_df["train_accuracy"], marker="o", label="train")
axes[0].plot(cv_results_df["C"], cv_results_df["cv_val_accuracy"], marker="o", label="cv validation")
axes[0].axvline(best_C, color="grey", linestyle="--", label=f"selected C={best_C}")
axes[0].set_xscale("log")
axes[0].set_xlabel("C (inverse regularization strength)")
axes[0].set_ylabel("Accuracy")
axes[0].set_title("Bias-variance trade-off across C")
axes[0].legend()

pipeline.set_params(clf__C=best_C)
train_sizes, train_scores, val_scores = learning_curve(
    pipeline, X_train, y_train, cv=cv, train_sizes=np.linspace(0.2, 1.0, 5),
    random_state=RUN_SEED,
)
axes[1].plot(train_sizes, train_scores.mean(axis=1), marker="o", label="train")
axes[1].plot(train_sizes, val_scores.mean(axis=1), marker="o", label="cv validation")
axes[1].set_xlabel("Training set size")
axes[1].set_ylabel("Accuracy")
axes[1].set_title(f"Learning curve (C={best_C})")
axes[1].legend()

plt.tight_layout()
plt.savefig(REPORTS_DIR / "overfitting_diagnostics.png", dpi=150)
plt.show()


## 14. Evaluation: Classification, Ranking, and Calibration

Fits the final model (selected `C`) on the full training split, checks the
held-out **validation** split (used only for threshold selection — never for
choosing `C`), then reports metrics on the **test** split exactly once.


In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay, accuracy_score, classification_report

pipeline.fit(X_train, y_train)

# Threshold comes from validation only — never from test.
val_probs = pipeline.predict_proba(X_val)[:, 1]
best_threshold = tune_threshold(y_val, val_probs)
print(f"Selected decision threshold from validation split: {best_threshold:.2f}")

test_probs = pipeline.predict_proba(X_test)[:, 1]
test_preds = test_probs >= best_threshold
test_df = test_df.copy()
test_df["learned_score"] = test_probs

learned_metrics = {
    "threshold": best_threshold,
    "precision": precision_score(y_test, test_preds, zero_division=0),
    "recall": recall_score(y_test, test_preds, zero_division=0),
    "f1": f1_score(y_test, test_preds, zero_division=0),
    "roc_auc": roc_auc_score(y_test, test_probs),
    "accuracy": accuracy_score(y_test, test_preds),
    **ranking_metrics(test_df, "learned_score"),
}

print(classification_report(y_test, test_preds))
print(json.dumps(learned_metrics, indent=2))
print(f"\nAccuracy {learned_metrics['accuracy']:.3f} against a Bayes ceiling of {bayes_ceiling:.3f} "
      f"({learned_metrics['accuracy'] / bayes_ceiling:.1%} of the achievable maximum).")

# Cold-start: unseen jobs. This is the scenario the product actually faces.
cold_probs = pipeline.predict_proba(cold_test_df[FEATURE_COLS].to_numpy())[:, 1]
cold_preds = cold_probs >= best_threshold
cold_test_df = cold_test_df.copy()
cold_test_df["learned_score"] = cold_probs
cold_metrics = {
    "precision": precision_score(cold_test_df["is_match"], cold_preds, zero_division=0),
    "recall": recall_score(cold_test_df["is_match"], cold_preds, zero_division=0),
    "f1": f1_score(cold_test_df["is_match"], cold_preds, zero_division=0),
    "roc_auc": roc_auc_score(cold_test_df["is_match"], cold_probs),
    "accuracy": accuracy_score(cold_test_df["is_match"], cold_preds),
    **ranking_metrics(cold_test_df, "learned_score"),
}
print("\nCold-start (unseen job postings):")
print(json.dumps(cold_metrics, indent=2))
print(f"warm->cold F1 drop: {learned_metrics['f1'] - cold_metrics['f1']:+.3f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
ConfusionMatrixDisplay.from_predictions(y_test, test_preds, ax=axes[0])
axes[0].set_title("Confusion matrix (warm test)")
RocCurveDisplay.from_predictions(y_test, test_probs, ax=axes[1], name="warm")
RocCurveDisplay.from_predictions(cold_test_df["is_match"], cold_probs, ax=axes[1], name="cold (new jobs)")
axes[1].set_title("ROC curve: warm vs cold start")

prob_true, prob_pred = calibration_curve(y_test, test_probs, n_bins=10)
axes[2].plot(prob_pred, prob_true, marker="o", label="model")
axes[2].plot([0, 1], [0, 1], linestyle="--", color="grey", label="perfectly calibrated")
axes[2].set_xlabel("Predicted probability")
axes[2].set_ylabel("Observed frequency")
axes[2].set_title("Calibration curve (warm test)")
axes[2].legend()

plt.tight_layout()
plt.savefig(REPORTS_DIR / "test_evaluation.png", dpi=150)
plt.show()

# Slice by job family, since that is how quality will actually be judged.
per_role_rows = []
for role, group in test_df.merge(jobs_df[["job_id", "title"]], on="job_id").groupby("title"):
    per_role_rows.append(
        {
            "role": role,
            "n": len(group),
            "precision": precision_score(group["is_match"], group["learned_score"] >= best_threshold, zero_division=0),
            "recall": recall_score(group["is_match"], group["learned_score"] >= best_threshold, zero_division=0),
        }
    )
display(pd.DataFrame(per_role_rows))


## 15. Recommendation Inference Pipeline (Top-k Jobs per Candidate)

Scores every same-role job for a sample of candidates and returns the top-k,
mirroring what `/api/match/results/{job_id}` will do in reverse (job → ranked
candidates) once Week 6 wires this into the backend.


In [ ]:
def recommend_top_k(candidate_row: pd.Series, k: int = 5) -> pd.DataFrame:
    role_jobs = jobs_by_role.get(candidate_row["target_role"])
    if role_jobs is None or role_jobs.empty:
        return pd.DataFrame()

    scored = []
    for _, job in role_jobs.iterrows():
        s_bert = float(
            np.dot(candidate_embeddings[candidate_row["embedding_idx"]], job_embeddings[job["embedding_idx"]])
        )
        s_graph = graph_overlap_score(candidate_row["skill_set"], job["required_skills"])
        s_growth = growth_score(
            len(candidate_row["certification_set"]), candidate_row["years_experience"], job["required_experience_years"]
        )
        final_score = float(pipeline.predict_proba([[s_bert, s_graph, s_growth]])[0, 1])
        missing_skills = set(job["required_skills"]) - candidate_row["skill_set"]
        scored.append(
            {
                "job_id": job["job_id"],
                "title": job["title"],
                "S_bert": s_bert,
                "S_graph": s_graph,
                "S_growth": s_growth,
                "final_score": final_score,
                "missing_skills": sorted(missing_skills),
            }
        )
    return pd.DataFrame(scored).sort_values("final_score", ascending=False).head(k)


sample_candidate = candidates_df.iloc[0]
recommendations = recommend_top_k(sample_candidate, k=5)
print(f"Top jobs for candidate {sample_candidate['candidate_id']} ({sample_candidate['target_role']}):")
display(recommendations)

recommendations.to_json(
    REPORTS_DIR / f"sample_recommendations_{sample_candidate['candidate_id']}.json",
    orient="records", indent=2,
)


## 16. Model Persistence, Batch Scoring, and Output Artifacts

Saves everything `app/core/ml_matcher.py` (Week 6) needs to reproduce this
exact scoring function without re-running the notebook:

- `hybrid_match_weights.json` — scaler mean/scale, logistic regression
  coefficients + intercept, feature order, chosen threshold, embedding model
  name/version, and the metrics this file was validated against.
- `hybrid_match_pipeline.joblib` — the fitted scikit-learn `Pipeline` itself,
  for exact reproducibility (the JSON is the human-readable/portable form).


In [ ]:
import joblib

scaler = pipeline.named_steps["scaler"]
clf = pipeline.named_steps["clf"]

artifact = {
    "model_version": MODEL_VERSION,
    "embedding_model": EMBEDDING_MODEL_NAME,
    "feature_order": FEATURE_COLS,
    "scaler_mean": scaler.mean_.tolist(),
    "scaler_scale": scaler.scale_.tolist(),
    "logistic_regression": {
        "coefficients": clf.coef_[0].tolist(),
        "intercept": float(clf.intercept_[0]),
        "C": float(best_C),
    },
    "decision_threshold": best_threshold,
    "label_scheme": {
        "type": "latent-driven Bernoulli outcome",
        "hidden_drivers": ["latent_competence", "latent_quality_bar", "latent_adaptability"],
        "noise_sd": NOISE_SD,
        "bayes_accuracy_ceiling": bayes_ceiling,
        "note": (
            "Labels are sampled from a probability that depends on hidden candidate/job "
            "traits the model never sees, so metrics are bounded by the Bayes ceiling "
            "and are not reproducible by thresholding the input features."
        ),
    },
    "test_metrics": learned_metrics,
    "cold_start_metrics": cold_metrics,
    "baseline_metrics": baseline_results,
    "trained_on_rows": {
        "train": len(train_df),
        "val": len(val_df),
        "test_warm": len(test_df),
        "test_cold": len(cold_test_df),
    },
}

(ARTIFACTS_DIR / "hybrid_match_weights.json").write_text(json.dumps(artifact, indent=2), encoding="utf-8")
joblib.dump(pipeline, ARTIFACTS_DIR / "hybrid_match_pipeline.joblib")

print(f"Artifacts written to {ARTIFACTS_DIR}:")
for f in sorted(ARTIFACTS_DIR.iterdir()):
    print(f" - {f.name}")


## 17. Auxiliary Check: Same Methodology on Kaggle Labeled Data

This is a **separate, optional** experiment — it does not feed into the
artifact saved above. It only runs if the Kaggle dataset was downloaded in
Section 2. Purpose: show the train/val/test-split + regularized-logistic-
regression + threshold-on-validation methodology also works on a dataset with
genuinely independent, real accept/reject labels, using whatever numeric
fields `users.csv` actually provides (not fabricated scores).


In [ ]:
if not KAGGLE_ENABLED:
    print("Kaggle dataset not available — skipping the auxiliary methodology check.")
else:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity

    id_col = "user_id"
    text_cols = [
        c for c in kaggle_users_df.columns
        if kaggle_users_df[c].dtype == object and c != id_col
    ]
    users_indexed = kaggle_users_df.set_index(id_col)
    users_indexed["_combined_text"] = users_indexed[text_cols].fillna("").agg(" ".join, axis=1)

    tfidf = TfidfVectorizer(max_features=2000, stop_words="english")
    tfidf_matrix = tfidf.fit_transform(users_indexed["_combined_text"])
    user_id_to_row = {uid: i for i, uid in enumerate(users_indexed.index)}

    aux_pairs = kaggle_feedback_df.dropna(subset=["user_id", "matched_user_id", "action"]).copy()
    aux_pairs = aux_pairs[
        aux_pairs["user_id"].isin(user_id_to_row) & aux_pairs["matched_user_id"].isin(user_id_to_row)
    ]

    text_sims = []
    personality_matches = []
    has_personality = "personality_type" in kaggle_users_df.columns
    for _, row in aux_pairs.iterrows():
        i, j = user_id_to_row[row["user_id"]], user_id_to_row[row["matched_user_id"]]
        text_sims.append(float(cosine_similarity(tfidf_matrix[i], tfidf_matrix[j])[0, 0]))
        if has_personality:
            personality_matches.append(
                int(users_indexed.iloc[i]["personality_type"] == users_indexed.iloc[j]["personality_type"])
            )

    aux_pairs["text_similarity"] = text_sims
    if has_personality:
        aux_pairs["personality_match"] = personality_matches
    aux_feature_cols = ["text_similarity"] + (["personality_match"] if has_personality else [])

    aux_group_split = GroupShuffleSplit(n_splits=1, test_size=0.4, random_state=RUN_SEED)
    aux_train_idx, aux_temp_idx = next(aux_group_split.split(aux_pairs, groups=aux_pairs["user_id"]))
    aux_temp = aux_pairs.iloc[aux_temp_idx]
    aux_val_test_split = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=RUN_SEED)
    aux_val_idx, aux_test_idx = next(aux_val_test_split.split(aux_temp, groups=aux_temp["user_id"]))

    aux_train, aux_val, aux_test = (
        aux_pairs.iloc[aux_train_idx], aux_temp.iloc[aux_val_idx], aux_temp.iloc[aux_test_idx],
    )

    aux_pipeline = Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(
        class_weight="balanced", max_iter=1000, random_state=RUN_SEED,
    ))])
    aux_pipeline.fit(aux_train[aux_feature_cols], aux_train["action"])

    aux_val_probs = aux_pipeline.predict_proba(aux_val[aux_feature_cols])[:, 1]
    aux_thresholds = np.linspace(0.1, 0.9, 33)
    aux_best_threshold = float(
        aux_thresholds[int(np.argmax([f1_score(aux_val["action"], aux_val_probs >= t) for t in aux_thresholds]))]
    )

    aux_test_probs = aux_pipeline.predict_proba(aux_test[aux_feature_cols])[:, 1]
    aux_test_preds = aux_test_probs >= aux_best_threshold
    aux_metrics = {
        "precision": precision_score(aux_test["action"], aux_test_preds, zero_division=0),
        "recall": recall_score(aux_test["action"], aux_test_preds, zero_division=0),
        "f1": f1_score(aux_test["action"], aux_test_preds, zero_division=0),
        "roc_auc": roc_auc_score(aux_test["action"], aux_test_probs) if aux_test["action"].nunique() > 1 else float("nan"),
        "n_train": len(aux_train), "n_val": len(aux_val), "n_test": len(aux_test),
    }
    print("Auxiliary Kaggle methodology check (generic profiles, real accept/reject labels):")
    print(json.dumps(aux_metrics, indent=2))


## 18. Optional CI Hook: Notebook Smoke Test

For CI, run this notebook headlessly against a small synthetic sample and
assert the artifact was produced with sane metrics — catches schema breaks or
metric regressions without needing a GPU runner. Example (not run here):

```bash
pip install nbclient
jupyter execute notebooks/hybrid_matching_model_training.ipynb \
  --ExecutePreprocessor.timeout=1800

python - <<'PY'
import json
from pathlib import Path

artifact = json.loads(Path("data_pipeline/artifacts/hybrid_match_weights.json").read_text())
metrics = artifact["test_metrics"]
assert metrics["f1"] > 0.0, "F1 regressed to zero — check the training pipeline"
assert metrics["roc_auc"] >= 0.5, "ROC-AUC at or below chance — model did not learn anything"
print("Notebook smoke test passed:", metrics)
PY
```

This is intentionally **not** wired into `.github/workflows/backend-ci.yml` yet
— it needs the small MiniLM model download, which is slower than the rest of
backend CI. Add it as a separate, allowed-to-be-slower workflow once the
notebook's outputs are trusted (Week 7).


## 19. Summary

- **Data**: synthetic Kenyan candidates/jobs across Data Analyst, DevOps Engineer,
  Software Engineer, Accountant and Business Analyst, each with one of three
  in-role *specialisations* so that text carries genuine semantic signal.
- **Features**: `S_bert` (MiniLM cosine similarity), `S_graph` (weighted skill
  overlap matching the Neo4j ontology seed), `S_growth`
  (experience/certification heuristic). Hidden latent traits are asserted out of
  the feature table.
- **Label**: a Bernoulli outcome driven partly by observable signal and partly by
  hidden traits plus Gaussian noise, so it cannot be reverse-engineered from the
  model's inputs and has a finite Bayes ceiling (printed in Section 9).
- **Model**: L2-regularized logistic regression; `C` chosen by 5-fold CV on the
  training split only, threshold chosen on validation only, metrics reported once
  on held-out tests.
- **Leakage controls**: grouped by `candidate_id`, *and* a fully held-out set of
  job postings for cold-start evaluation.
- **Fair baselines**: every baseline threshold is tuned on validation by the same
  procedure as the learned model.

### How to read the results honestly

- Compare accuracy against the **Bayes ceiling**, not against 1.0. Scoring ~0.80
  against a ceiling of ~0.85 is a strong result; the gap to 1.0 is label noise
  no model can remove.
- Check the **warm → cold F1 drop**. A small drop means the model learned
  transferable matching; a large drop means it memorised specific postings.
- If `S_bert`'s coefficient is still near zero now that specialisations vary the
  text, that is a real finding worth reporting — it would mean skill-graph
  overlap and experience fit carry the signal in this domain.

### Next steps (Week 6 backend integration)

1. `app/core/bert_embedder.py` loads the same `EMBEDDING_MODEL_NAME` at startup.
2. `app/core/graph_service.py` computes `S_graph` from live Neo4j queries using
   the `RELATED_TO` weights seeded in `backend/app/db/neo4j_db.py`.
3. `app/core/ml_matcher.py` loads `hybrid_match_weights.json`, standardises
   `[S_bert, S_graph, S_growth]` with the saved mean/scale, applies the saved
   coefficients and intercept, and thresholds at `decision_threshold`. Load the
   **JSON**, not the `.joblib` — the pickle is untracked precisely so nothing
   deserialises it at runtime.
4. Re-run this notebook whenever the generator changes or real BrighterMonday
   postings land, then replace the artifact.
